# Logistic Regression Code-Along

Implement the code-blocks below in order to implement the logistic regressor. We will be using the `breast_cancer` toy dataset which we can directly load from sklearn.

In [1]:
from sklearn.datasets import load_breast_cancer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import time

In [2]:
# load dataset
X, y = load_breast_cancer(return_X_y=True, as_frame=True)

# view first 5 rows of predictors
X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [3]:
# view first 5 random samples of target data
y.sample(5)

155    1
265    0
115    1
188    1
436    1
Name: target, dtype: int64

In [4]:
# TODO: select only the "mean radius", "mean texture", and "mean perimeter" predictors for your predictors
X = X[['mean radius', 'mean texture', 'mean perimeter']]

# TODO: split into test and training sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# view first 5 rows of training data
X_train.head(5)

,mean radius,mean texture,mean perimeter
68,9.029,17.33,58.79
181,21.090,26.57,142.70
63,9.173,13.86,59.20
248,10.650,25.22,68.01
60,10.170,14.88,64.55


In [5]:
# Randomly search for the best hyperparameters on a logistic regression model
param_dist = {
    'penalty': ['l1', 'l2'],
    'C': np.linspace(0.01, 1, 100),
    'solver': ['saga'], 
    'max_iter': [10000]
}

random_search = RandomizedSearchCV(LogisticRegression(), param_distributions=param_dist, cv=5, scoring='accuracy', random_state=42)
random_search.fit(X_train, y_train)

# Best model from random search
best_params_random = random_search.best_params_
best_score_random = random_search.best_score_

print(f"RandomizedSearchCV - Best Params: {best_params_random}")
print(f"RandomizedSearchCV - Cross-Val Accuracy: {best_score_random:.2f}")

/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed 

RandomizedSearchCV - Best Params: {'solver': 'saga', 'penalty': 'l1', 'max_iter': 10000, 'C': np.float64(0.8)}
RandomizedSearchCV - Cross-Val Accuracy: 0.89


/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [6]:
# Use the best model found from RandomizedSearchCV to predict on unseen test data

# extract the best estimator
best_log = random_search.best_estimator_

# predict on testing data
log_predictions = best_log.predict(X_test)

# evaluate its accuracy
test_score = accuracy_score(log_predictions, y_test)

print(f"RandomizedSearchCV - Coefficients: {best_log.coef_}")
print(f"RandomizedSearchCV - Test Accuracy: {test_score:.2f}")

RandomizedSearchCV - Coefficients: [[ 3.22442342  0.0138055  -0.537248  ]]
RandomizedSearchCV - Test Accuracy: 0.89


The coefficients above relate to the predictors "mean radius", "mean texture", and "mean perimeter."

Note that positive values indicate a higher log-odds that a tumor is malignant. However, negative values indicate a lower log-odds that a tumor is malignant.

Which predictor seems to indicate a higher probability that a tumor is malignant? 

In [7]:
# Now we will see why we prefer to use RandomizedSearchCV
# Search the grid for the best hyperparameters on a logistic regression model (this will take at least 3 minutes!)
grid_search = GridSearchCV(LogisticRegression(), param_grid=param_dist, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# Best model from grid search
best_params_grid = grid_search.best_params_
best_score_grid = grid_search.best_score_

print(f"GridSearchCV - Best Params: {best_params_grid}")
print(f"GridSearchCV - Cross-Val Accuracy: {best_score_grid:.2f}")

/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', 

GridSearchCV - Best Params: {'C': np.float64(0.73), 'max_iter': 10000, 'penalty': 'l1', 'solver': 'saga'}
GridSearchCV - Cross-Val Accuracy: 0.89


/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


# Challenge

Create a logistic regressor on the `hotel` dataset. Your target variable will be `is_canceled` column, while your precictor will be the `lead_time` column.

After creating your model extract best hyperparameters using both `RandomizedSearchCV` and `GridSearchCV`. Evaluate the coefficients and the accuracy of both hyperparameter finding algorithms (we will learn more about classification accuracy metrics tomorrow). 

Answer the analytical questions listed below as well.

In [10]:
# TODO: Implement the logistic regression model on the `hotel.csv` dataset

# load dataset
df = pd.read_csv('hotel.csv')

df.head(5)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [17]:
# Assigning my target variable:
y = df['is_canceled']

# Assigning my predictor:
X = df[['lead_time']]

In [18]:
# view first 5 random samples of target data
y.sample(5)

117655    0
96983     0
111835    0
81349     0
111400    0
Name: is_canceled, dtype: int64

In [19]:
# split into test and training sets:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# view first 5 rows of training data
X_train.head(5)

,lead_time
67702,64
115851,34
57345,8
11622,251
33333,23


In [20]:
# Randomly search for the best hyperparameters on a logistic regression model
param_dist = {
    'penalty': ['l1', 'l2'],
    'C': np.linspace(0.01, 1, 100),
    'solver': ['saga'], 
    'max_iter': [10000]
}

random_search = RandomizedSearchCV(LogisticRegression(), param_distributions=param_dist, cv=5, scoring='accuracy', random_state=42)
random_search.fit(X_train, y_train)

# Best model from random search
best_params_random = random_search.best_params_
best_score_random = random_search.best_score_

print(f"RandomizedSearchCV - Best Params: {best_params_random}")
print(f"RandomizedSearchCV - Cross-Val Accuracy: {best_score_random:.2f}")

/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed 

RandomizedSearchCV - Best Params: {'solver': 'saga', 'penalty': 'l2', 'max_iter': 10000, 'C': np.float64(0.48000000000000004)}
RandomizedSearchCV - Cross-Val Accuracy: 0.66


In [21]:
# Use the best model found from RandomizedSearchCV to predict on unseen test data

# extract the best estimator
best_log = random_search.best_estimator_

# predict on testing data
log_predictions = best_log.predict(X_test)

# evaluate its accuracy
test_score = accuracy_score(log_predictions, y_test)

print(f"RandomizedSearchCV - Coefficients: {best_log.coef_}")
print(f"RandomizedSearchCV - Test Accuracy: {test_score:.2f}")

RandomizedSearchCV - Coefficients: [[0.00582172]]
RandomizedSearchCV - Test Accuracy: 0.66


In [22]:
# Now we will see why we prefer to use RandomizedSearchCV
# Search the grid for the best hyperparameters on a logistic regression model (this will take at least 3 minutes!)
grid_search = GridSearchCV(LogisticRegression(), param_grid=param_dist, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# Best model from grid search
best_params_grid = grid_search.best_params_
best_score_grid = grid_search.best_score_

print(f"GridSearchCV - Best Params: {best_params_grid}")
print(f"GridSearchCV - Cross-Val Accuracy: {best_score_grid:.2f}")

/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/opt/miniconda3/envs/ds_class/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', 

GridSearchCV - Best Params: {'C': np.float64(0.01), 'max_iter': 10000, 'penalty': 'l1', 'solver': 'saga'}
GridSearchCV - Cross-Val Accuracy: 0.66


## Writeup

Answer the analytical questions below using the metrics you've calculated.

1) What was the accuracy of your hyperparameters extracted from `GridSearchCV`? Was this any different from `RandomizedSearchCV`? Did the extended run-time of `GridSearchCV` provide huge increases in accuracy?

...

2) According to your coefficients, how did `lead_time` influence the probability that a booking would be cancelled? 

...
